In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

C:\Users\UDAY MAURYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\UDAY MAURYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\UDAY MAURYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\

In [2]:
data = pd.read_csv("creditcard.csv")
print(data.head())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [3]:
# 0 = Normal, 1 = Fraud
print(data['Class'].value_counts())

Class
0    284315
1       492
Name: count, dtype: int64


In [4]:
X = data.drop('Class', axis=1)
y = data['Class']

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

In [7]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.0017,   # approximate fraud rate
    random_state=42
)

iso_forest.fit(X_train)

,n_estimators,100
,max_samples,'auto'
,contamination,0.0017
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [8]:
y_pred_iso = iso_forest.predict(X_test)

# Convert output
# -1 → Fraud, 1 → Normal
y_pred_iso = np.where(y_pred_iso == -1, 1, 0)

In [9]:
print(confusion_matrix(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso))
print("ROC AUC:", roc_auc_score(y_test, y_pred_iso))

[[85201    94]
 [  111    37]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.28      0.25      0.27       148

    accuracy                           1.00     85443
   macro avg       0.64      0.62      0.63     85443
weighted avg       1.00      1.00      1.00     85443

ROC AUC: 0.6244489712175392


In [10]:
X_train_normal = X_train[y_train == 0]

In [11]:
# Autoencoder architecture
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoder = Dense(32, activation="relu")(input_layer)
encoder = Dense(16, activation="relu")(encoder)

decoder = Dense(32, activation="relu")(encoder)
decoder = Dense(input_dim, activation="linear")(decoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer="adam", loss="mse")

In [12]:
history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    shuffle=True
)

Epoch 1/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.5392 - val_loss: 0.3426
Epoch 2/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.2892 - val_loss: 0.2500
Epoch 3/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.2348 - val_loss: 0.2188
Epoch 4/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.2093 - val_loss: 0.2127
Epoch 5/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.1934 - val_loss: 0.1871
Epoch 6/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.1810 - val_loss: 0.1917
Epoch 7/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.1683 - val_loss: 0.1602
Epoch 8/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.1555 - val_loss: 0.1513
Epoch 9/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.1450 - val_loss: 0.1363
Epoch 10/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.1325 - val_loss: 0.1233
Epoch 11/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.1197 - val_loss: 0.1122
Epoch 12/20
700/700 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/st

In [13]:
reconstructions = autoencoder.predict(X_test)
mse = np.mean(np.square(X_test - reconstructions), axis=1)

2671/2671 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step


In [14]:
threshold = np.percentile(mse, 95)

In [15]:
y_pred_ae = (mse > threshold).astype(int)

In [16]:
print(confusion_matrix(y_test, y_pred_ae))
print(classification_report(y_test, y_pred_ae))
print("ROC AUC:", roc_auc_score(y_test, y_pred_ae))

[[81145  4150]
 [   25   123]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97     85295
           1       0.03      0.83      0.06       148

    accuracy                           0.95     85443
   macro avg       0.51      0.89      0.52     85443
weighted avg       1.00      0.95      0.97     85443

ROC AUC: 0.8912132059957255


In [17]:
alerts = pd.DataFrame({
    "FraudScore": mse,
    "Alert": y_pred_ae
})

alerts["RiskLevel"] = pd.cut(
    alerts["FraudScore"],
    bins=[0, threshold, threshold*1.5, mse.max()],
    labels=["Low", "Medium", "High"]
)

print(alerts.head())

   FraudScore  Alert RiskLevel
0    0.098473      0       Low
1    0.013797      0       Low
2    0.073500      0       Low
3    0.105239      0       Low
4    0.029867      0       Low
